In [8]:
import os
print(os.getcwd())

c:\Users\ANUJI THRIMANNA\Downloads\airbnb-assessment\notebooks


In [ ]:
## 1. Load Data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import geopandas as gpd

# Load sample data for NYC
nyc_listings = pd.read_csv('../data/raw/nyc/listings.csv.gz')
nyc_calendar = pd.read_csv('../data/raw/nyc/calendar.csv.gz')
nyc_reviews = pd.read_csv('../data/raw/nyc/reviews.csv.gz')
nyc_neighbourhoods = gpd.read_file(
    '../data/raw/nyc/neighbourhoods.geojson'
)

print(f"Listings: {len(nyc_listings):,} rows, {len(nyc_listings.columns)} columns")
print(f"Calendar: {len(nyc_calendar):,} rows, {len(nyc_calendar.columns)} columns")
print(f"Reviews: {len(nyc_reviews):,} rows, {len(nyc_reviews.columns)} columns")
print(f"Neighbourhoods: {len(nyc_neighbourhoods):,} rows")

Listings: 35,036 rows, 90 columns
Calendar: 12,788,141 rows, 5 columns
Reviews: 1,003,299 rows, 6 columns
Neighbourhoods: 233 rows


In [ ]:
## 2. Schema Documentation

def document_schema(df, name):
    """Create detailed schema documentation for a DataFrame."""
    schema_doc = {
        'table_name': name,
        'row_count': len(df),
        'columns': []
    }
    
    for col in df.columns:
        col_info = {
            'name': col,
            'dtype': str(df[col].dtype),
            'nulls': df[col].isna().sum(),
            'null_pct': (df[col].isna().sum() / len(df)) * 100,
            'unique': df[col].nunique(),
            'sample': df[col].dropna().head(3).tolist()
        }
        schema_doc['columns'].append(col_info)
    
    return pd.DataFrame(schema_doc['columns'])

# Document all tables
for name, df in [
    ('listings', nyc_listings),
    ('calendar', nyc_calendar),
    ('reviews', nyc_reviews)
]:
    print(f"\n{'='*60}")
    print(f"SCHEMA: {name.upper()}")
    print(f"{'='*60}")
    schema_df = document_schema(df, name)
    print(schema_df.to_string())


SCHEMA: LISTINGS
                                            name    dtype  nulls    null_pct  unique                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [11]:
## 3. Key Columns & Interpretations

# Document key columns and their meanings
key_columns = {
    'listings': {
        'id': 'Unique listing identifier (primary key)',
        'host_id': 'Host identifier (foreign key to host)',
        'price': 'Nightly price in USD (string, needs cleaning)',
        'neighbourhood': 'Neighbourhood name',
        'latitude': 'Latitude coordinate',
        'longitude': 'Longitude coordinate',
        'room_type': 'Type of listing (Entire home/apt, Private room, Shared room)',
        'minimum_nights': 'Minimum nights required for booking',
        'availability_365': 'Days available in next 365 days',
        'number_of_reviews': 'Total number of reviews',
        'reviews_per_month': 'Average reviews per month',
        'calculated_host_listings_count': 'Number of listings the host has'
    },
    'calendar': {
        'listing_id': 'Listing identifier (foreign key)',
        'date': 'Date of calendar entry',
        'available': 'Availability flag (t/f)',
        'price': 'Price for that date (string, needs cleaning)'
    },
    'reviews': {
        'listing_id': 'Listing identifier (foreign key)',
        'id': 'Review identifier',
        'date': 'Review date',
        'reviewer_id': 'Reviewer identifier',
        'comments': 'Review text'
    }
}

for table, columns in key_columns.items():
    print(f"\n{table.upper()} - Column Interpretations:")
    for col, meaning in columns.items():
        print(f"  {col}: {meaning}")


LISTINGS - Column Interpretations:
  id: Unique listing identifier (primary key)
  host_id: Host identifier (foreign key to host)
  price: Nightly price in USD (string, needs cleaning)
  neighbourhood: Neighbourhood name
  latitude: Latitude coordinate
  longitude: Longitude coordinate
  room_type: Type of listing (Entire home/apt, Private room, Shared room)
  minimum_nights: Minimum nights required for booking
  availability_365: Days available in next 365 days
  number_of_reviews: Total number of reviews
  reviews_per_month: Average reviews per month
  calculated_host_listings_count: Number of listings the host has

CALENDAR - Column Interpretations:
  listing_id: Listing identifier (foreign key)
  date: Date of calendar entry
  available: Availability flag (t/f)
  price: Price for that date (string, needs cleaning)

REVIEWS - Column Interpretations:
  listing_id: Listing identifier (foreign key)
  id: Review identifier
  date: Review date
  reviewer_id: Reviewer identifier
  commen

In [12]:
## 4. Primary Key & Foreign Key Relationships

# Document relationships
print("\n" + "="*60)
print("DATA RELATIONSHIPS")
print("="*60)
print("""
┌─────────────────────────────────────────────────────────────────────┐
│                       RELATIONSHIP DIAGRAM                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────────┐            ┌─────────────┐                       │
│  │  LISTINGS   │            │  CALENDAR   │                       │
│  │  (PK: id)   │◄───────────│  (FK: list- │                       │
│  │             │            │   ing_id)   │                       │
│  └──────┬──────┘            └─────────────┘                       │
│         │                                                          │
│         │                                                          │
│         │                                                          │
│         ▼                                                          │
│  ┌─────────────┐            ┌─────────────────┐                  │
│  │  REVIEWS    │            │ NEIGHBOURHOODS  │                  │
│  │  (FK: list- │            │ (PK: neighbour-  │                  │
│  │   ing_id)   │            │  hood_name)      │                  │
│  └─────────────┘            └─────────────────┘                  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

Key Joins:
1. listings.id = calendar.listing_id  (1-to-many)
2. listings.id = reviews.listing_id   (1-to-many)
3. listings.neighbourhood = neighbourhoods.neighbourhood_name  (many-to-1)
""")


DATA RELATIONSHIPS

┌─────────────────────────────────────────────────────────────────────┐
│                       RELATIONSHIP DIAGRAM                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────────┐            ┌─────────────┐                       │
│  │  LISTINGS   │            │  CALENDAR   │                       │
│  │  (PK: id)   │◄───────────│  (FK: list- │                       │
│  │             │            │   ing_id)   │                       │
│  └──────┬──────┘            └─────────────┘                       │
│         │                                                          │
│         │                                                          │
│         │                                                          │
│         ▼                                                          │
│  ┌─────────────┐            ┌─────────────────┐        

In [13]:
## 5. Data Quality Assessment

def assess_data_quality(df, name):
    """Assess data quality issues in a dataset."""
    issues = []
    
    # Check null rates
    for col in df.columns:
        null_pct = (df[col].isna().sum() / len(df)) * 100
        if null_pct > 20:
            issues.append(f"High null rate in {col}: {null_pct:.1f}%")
        elif null_pct > 10:
            issues.append(f"Moderate null rate in {col}: {null_pct:.1f}%")
    
    # Check duplicates
    if 'id' in df.columns:
        dupes = df['id'].duplicated().sum()
        if dupes > 0:
            issues.append(f"Duplicate IDs found: {dupes:,}")
    
    # Check numeric outliers
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
        if outliers > 0:
            issues.append(f"Outliers in {col}: {outliers:,} rows ({outliers/len(df)*100:.1f}%)")
    
    return issues

# Check all datasets
for name, df in [
    ('listings', nyc_listings),
    ('calendar', nyc_calendar),
    ('reviews', nyc_reviews)
]:
    print(f"\n{name.upper()} Quality Issues:")
    issues = assess_data_quality(df, name)
    if issues:
        for issue in issues:
            print(f"  ⚠️ {issue}")
    else:
        print("  ✅ No major quality issues found")


LISTINGS Quality Issues:
  ⚠️ High null rate in neighborhood_overview: 100.0%
  ⚠️ High null rate in host_since: 100.0%
  ⚠️ High null rate in host_location: 22.9%
  ⚠️ High null rate in host_about: 43.9%
  ⚠️ High null rate in host_response_time: 100.0%
  ⚠️ High null rate in host_response_rate: 100.0%
  ⚠️ High null rate in host_acceptance_rate: 100.0%
  ⚠️ High null rate in host_thumbnail_url: 100.0%
  ⚠️ High null rate in host_neighbourhood: 100.0%
  ⚠️ High null rate in host_total_listings_count: 100.0%
  ⚠️ High null rate in host_verifications: 100.0%
  ⚠️ High null rate in neighbourhood: 100.0%
  ⚠️ High null rate in bathrooms: 89.2%
  ⚠️ High null rate in bedrooms: 34.4%
  ⚠️ High null rate in beds: 89.3%
  ⚠️ High null rate in price: 40.9%
  ⚠️ High null rate in price_quote_checkin_date: 37.4%
  ⚠️ High null rate in price_quote_checkout_date: 37.4%
  ⚠️ High null rate in price_quote_total_price: 40.9%
  ⚠️ High null rate in price_quote_price_per_night: 40.9%
  ⚠️ High null ra

In [14]:
## 6. City Comparison - Initial Observations

# Compare schemas across cities
cities = ['nyc', 'bcn', 'edi']
city_data = {}

for city in cities:
    city_path = Path(f'data/raw/{city}')
    if city_path.exists():
        listings_path = city_path / 'listings.csv'
        if listings_path.exists():
            df = pd.read_csv(listings_path)
            city_data[city] = {
                'rows': len(df),
                'columns': len(df.columns),
                'cols': set(df.columns)
            }

print("\nCity Comparison:")
for city, info in city_data.items():
    print(f"\n{city.upper()}:")
    print(f"  Rows: {info['rows']:,}")
    print(f"  Columns: {info['columns']}")
    
# Check for structural consistency
if len(city_data) > 1:
    common_cols = set.intersection(*[info['cols'] for info in city_data.values()])
    print(f"\nCommon columns across all cities: {len(common_cols)}")
    print(f"  {sorted(common_cols)[:10]}...")
    
    # Note differences
    for city, info in city_data.items():
        unique_cols = info['cols'] - common_cols
        if unique_cols:
            print(f"\n  {city.upper()} unique columns: {unique_cols}")

print("\n✓ Schema documentation complete")


City Comparison:

✓ Schema documentation complete
